# Notebook 4: Word Filters — Exact-Match Blocklists for Insurance Compliance

## Amazon Bedrock Guardrails

This notebook adds word filter policies to the guardrail built in Notebooks 1-3. Word filters are the simplest guardrail mechanism — pure exact-match blocklists with no classifiers or models involved. If the word or phrase appears in the input or output, it's caught.

### What This Notebook Covers
- Profanity filters using Bedrock's managed word list
- Custom word blocklists for compliance-flagged phrases ("guaranteed coverage", "we promise")
- Blocking competitor names and internal codenames
- Combining all four policy layers into a single guardrail

### Key Concept
Content filters, denied topics, and PII detection all use classifiers that understand context and intent. Word filters don't — they're a simple string match. Think of it as the difference between a drug-sniffing dog (classifier) and a "no liquids over 100ml" sign (word filter). The sign doesn't need to understand what the liquid is — if it matches the rule, it's caught.

### Prerequisites
- Notebooks 1-3 completed (guardrail with content filters + denied topics + PII detection)
- Guardrail ID from previous notebooks

## 1. Setup & Retrieve Existing Guardrail

Connect to the guardrail from Notebooks 1-3 and verify that all three existing policy layers are in place before adding word filters.

In [72]:
import boto3
import json
import random
import string
from datetime import datetime

# Control plane — create and manage guardrails
bedrock = boto3.client('bedrock', region_name='us-east-1')

# Data plane — invoke models with guardrails applied
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

MODEL_ID = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

# Guardrail ID from Notebooks 1-3 — replace with your actual ID
GUARDRAIL_ID = 'your-guardrail-id'
GUARDRAIL_VERSION = 'DRAFT'

# Verify the guardrail and show all existing policies
guardrail = bedrock.get_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion=GUARDRAIL_VERSION
)

print(f"Connected to guardrail: {guardrail['name']}")
print(f"Status: {guardrail['status']}")

print(f"\nContent Filters:")
print(f"{'Category':<16} {'Input':<10} {'Output':<10}")
print(f"{'-'*36}")
for f in guardrail['contentPolicy']['filters']:
    print(f"{f['type']:<16} {f['inputStrength']:<10} {f['outputStrength']:<10}")

print(f"\nDenied Topics:")
for topic in guardrail['topicPolicy']['topics']:
    print(f"  - {topic['name']}")

print(f"\nPII Detectors: {len(guardrail['sensitiveInformationPolicy']['piiEntities'])} built-in")
print(f"Custom Regex:  {len(guardrail['sensitiveInformationPolicy']['regexes'])} patterns")

Connected to guardrail: insurance-assistant-guardrail
Status: READY

Content Filters:
Category         Input      Output    
------------------------------------
VIOLENCE         LOW        MEDIUM    
PROMPT_ATTACK    HIGH       NONE      
MISCONDUCT       MEDIUM     HIGH      
HATE             HIGH       HIGH      
SEXUAL           HIGH       HIGH      
INSULTS          LOW        HIGH      

Denied Topics:
  - Investment Advice
  - Medical Diagnosis
  - Legal Advice
  - Coverage Guarantees
  - Competitor Comparisons
  - Claim Value Adjustments

PII Detectors: 10 built-in
Custom Regex:  3 patterns


## 2. Understanding Word Filters

Word filters are the simplest guardrail mechanism — no classifiers, no models, no intelligence. Pure exact-match string matching against a blocklist. If the word or phrase appears in the text, it's caught.

Bedrock provides two types:

### Managed Word List
A built-in profanity filter maintained by AWS. Toggle it on and it handles profanity across multiple languages. You don't see or manage the list — it's a black box.

### Custom Word List
Your own blocklist of words and phrases. Three categories for insurance:

| Category | Examples | Why |
|----------|----------|-----|
| **Compliance-flagged phrases** | "guaranteed coverage", "we promise", "100% covered" | Misrepresentation risk — only underwriters can make guarantees |
| **Competitor names** | Specific insurer names | Prevent the model from mentioning competitors by name in responses |
| **Internal codenames** | Project names, system names | Prevent leaking internal terminology to customers |

In [73]:
# Managed profanity filter — AWS maintains this list
# We just toggle it on

managed_word_list = {'managedWordListsConfig': [{'type': 'PROFANITY'}]}

# Custom word lists for insurance domain
custom_words = [
    # Compliance-flagged phrases — misrepresentation risk
    'guaranteed coverage',
    'we promise',
    'we guarantee',
    '100% covered',
    'definitely covered',
    'coverage is guaranteed',
    'claim will be approved',
    'you are fully covered',
    
    # Competitor names — prevent model from mentioning in responses
    'State Farm',
    'Geico',
    'Allstate',
    'Progressive',
    'Intact Insurance',
    'Sun Life',
    'Manulife',
    'Desjardins Insurance',
    
    # Internal codenames — prevent leaking to customers
    'Project Lighthouse',
    'ATLAS-7',
    'ClaimsEngine v3',
    'Operation Maple'
]

print(f"Managed word list: PROFANITY enabled")
print(f"\nCustom blocked words/phrases: {len(custom_words)}")
print(f"\nCompliance phrases:")
for w in custom_words[:8]:
    print(f"  - \"{w}\"")
print(f"\nCompetitor names:")
for w in custom_words[8:18]:
    print(f"  - \"{w}\"")
print(f"\nInternal codenames:")
for w in custom_words[18:]:
    print(f"  - \"{w}\"")

Managed word list: PROFANITY enabled

Custom blocked words/phrases: 20

Compliance phrases:
  - "guaranteed coverage"
  - "we promise"
  - "we guarantee"
  - "100% covered"
  - "definitely covered"
  - "coverage is guaranteed"
  - "claim will be approved"
  - "you are fully covered"

Competitor names:
  - "State Farm"
  - "Geico"
  - "Allstate"
  - "Progressive"
  - "Intact Insurance"
  - "Sun Life"
  - "Manulife"
  - "Desjardins Insurance"
  - "Project Lighthouse"
  - "ATLAS-7"

Internal codenames:
  - "ClaimsEngine v3"
  - "Operation Maple"


## 3. Update the Guardrail with Word Filters

Adding the word filter policy alongside the existing three layers. This completes all four policy types on a single guardrail — content filters, denied topics, PII detection, and word filters.

In [74]:
# Denied topics from Notebook 2 — needed for full config update

denied_topics = [
    {
        'name': 'Investment Advice',
        'definition': (
            'Advice about investing money including stocks, bonds, mutual funds, '
            'ETFs, retirement accounts, market timing, or portfolio allocation.'
        ),
        'examples': [
            'Should I invest my insurance payout in index funds?',
            'What stocks should I buy with my settlement money?',
            'Is now a good time to put money in the market?',
            'How much of my payout should I put in a TFSA?',
        ],
        'type': 'DENY'
    },
    {
        'name': 'Medical Diagnosis',
        'definition': (
            'Providing a medical diagnosis, interpreting what symptoms mean, '
            'or recommending specific treatments or medications. Does not include '
            'discussing symptoms for claim documentation purposes.'
        ),
        'examples': [
            'I have back pain after the accident — do I have a herniated disc?',
            'Should I get an MRI or is physiotherapy enough?',
            'What medication should I take for whiplash?',
            'Is my headache a sign of a concussion?',
        ],
        'type': 'DENY'
    },
    {
        'name': 'Legal Advice',
        'definition': (
            'Legal opinions or strategy about suing, hiring lawyers, or accepting '
            'settlements. Does not include questions about the company\'s internal '
            'dispute or appeals processes.'
        ),
        'examples': [
            'Should I sue the other driver?',
            'Is this settlement offer fair or should I fight it?',
            'Can I take legal action against my insurance company?',
            'What are my legal rights if my claim is denied?',
        ],
        'type': 'DENY'
    },
    {
        'name': 'Coverage Guarantees',
        'definition': (
            'Requests to guarantee, promise, or confirm coverage outcomes, '
            'claim approvals, or payout amounts.'
        ),
        'examples': [
            'Will my claim definitely be approved?',
            'Can you guarantee my roof replacement is covered?',
            'Promise me this will be paid out within 30 days',
            'Confirm that my policy covers this accident 100%',
        ],
        'type': 'DENY'
    },
    {
        'name': 'Competitor Comparisons',
        'definition': (
            'Evaluating or ranking the company against competitor insurers. '
            'Includes recommending switching to a competitor or stating which '
            'insurer is better, cheaper, or offers superior coverage.'
        ),
        'examples': [
            'Is your auto insurance better than State Farm?',
            'Should I switch to Geico for a lower premium?',
            'How does your coverage compare to Allstate?',
            'My friend says Progressive is cheaper — is that true?',
        ],
        'type': 'DENY'
    },
    {
        'name': 'Claim Value Adjustments',
        'definition': (
            'Requests to change, increase, or override a claim payout amount '
            'or adjuster decision. Does not include asking about the current '
            'status or value of a claim.'
        ),
        'examples': [
            'Can you increase my claim payout to cover the full repair cost?',
            'The adjuster undervalued my car — change it to $15,000',
            'Override the damage assessment and give me the full amount',
            'Adjust my claim value to match the dealer quote',
        ],
        'type': 'DENY'
    }
]

print(f"Loaded {len(denied_topics)} denied topics")

Loaded 6 denied topics


In [75]:
denied_topics[4] = {
    'name': 'Competitor Comparisons',
    'definition': (
        'Evaluating or ranking the company against competitor insurers. '
        'Includes stating which insurer is better or cheaper. Does not '
        'include mentioning a competitor when switching or transferring.'
    ),
    'examples': [
        'Which insurance company has the best rates?',
        'Are your premiums lower than other insurers?',
        'Why should I choose you over another company?',
        'Rate your coverage versus the competition',
    ],
    'type': 'DENY'
}

print(f"Updated: {denied_topics[4]['name']} ({len(denied_topics[4]['definition'])} chars)")

Updated: Competitor Comparisons (188 chars)


In [76]:
denied_topics[3] = {
    'name': 'Coverage Guarantees',
    'definition': (
        'Asking the system to make a binding promise or guarantee about a '
        'claim outcome or payout. Words like guarantee, promise, confirm, '
        'or definitely in the context of claim approval.'
    ),
    'examples': [
        'Guarantee that my claim will be approved',
        'Promise me you will pay this claim in full',
        'I need you to confirm my payout amount right now',
        'Tell me for certain this claim will definitely be paid',
    ],
    'type': 'DENY'
}

print(f"Updated: {denied_topics[3]['name']} ({len(denied_topics[3]['definition'])} chars)")

Updated: Coverage Guarantees (177 chars)


In [77]:
# Updated custom words — competitor names removed
# Competitor blocking handled by denied topics + system prompt instead

custom_words = [
    # Compliance-flagged phrases — misrepresentation risk
    'guaranteed coverage',
    'we promise',
    'we guarantee',
    '100% covered',
    'definitely covered',
    'coverage is guaranteed',
    'claim will be approved',
    'you are fully covered',
    
    # Internal codenames — prevent leaking to customers
    'Project Lighthouse',
    'ATLAS-7',
    'ClaimsEngine v3',
    'Operation Maple'
]

print(f"Updated custom blocked words/phrases: {len(custom_words)}")
print(f"\nCompliance phrases:")
for w in custom_words[:8]:
    print(f"  - \"{w}\"")
print(f"\nInternal codenames:")
for w in custom_words[8:]:
    print(f"  - \"{w}\"")
print(f"\nNote: Competitor names removed — handled by denied topics instead")

Updated custom blocked words/phrases: 12

Compliance phrases:
  - "guaranteed coverage"
  - "we promise"
  - "we guarantee"
  - "100% covered"
  - "definitely covered"
  - "coverage is guaranteed"
  - "claim will be approved"
  - "you are fully covered"

Internal codenames:
  - "Project Lighthouse"
  - "ATLAS-7"
  - "ClaimsEngine v3"
  - "Operation Maple"

Note: Competitor names removed — handled by denied topics instead


In [78]:
# PII config from Notebook 3 — needed for full config update

pii_config = [
    {'type': 'US_SOCIAL_SECURITY_NUMBER', 'action': 'BLOCK'},
    {'type': 'EMAIL', 'action': 'ANONYMIZE'},
    {'type': 'PHONE', 'action': 'ANONYMIZE'},
    {'type': 'NAME', 'action': 'ANONYMIZE'},
    {'type': 'CREDIT_DEBIT_CARD_NUMBER', 'action': 'BLOCK'},
    {'type': 'US_INDIVIDUAL_TAX_IDENTIFICATION_NUMBER', 'action': 'BLOCK'},
    {'type': 'URL', 'action': 'ANONYMIZE'},
    {'type': 'IP_ADDRESS', 'action': 'ANONYMIZE'},
    {'type': 'CA_SOCIAL_INSURANCE_NUMBER', 'action': 'BLOCK'},
    {'type': 'CA_HEALTH_NUMBER', 'action': 'BLOCK'}
]

regex_config = [
    {
        'name': 'Policy Number',
        'description': 'Company policy numbers in format POL-YYYY-XX-NNNNNN',
        'pattern': r'POL-\d{4}-[A-Z]{2}-\d{4,6}',
        'action': 'ANONYMIZE'
    },
    {
        'name': 'Claim Reference',
        'description': 'Claim reference numbers in format CLM-XX-YYYY-NNNNNN',
        'pattern': r'CLM-[A-Z]{2}-\d{4}-\d{4,6}',
        'action': 'ANONYMIZE'
    },
    {
        'name': 'Agent Code',
        'description': 'Internal agent/broker codes in format AGT-NNNNN',
        'pattern': r'AGT-\d{5}',
        'action': 'ANONYMIZE'
    }
]

print(f"Loaded {len(pii_config)} PII detectors and {len(regex_config)} custom regex patterns")

Loaded 10 PII detectors and 3 custom regex patterns


In [92]:
response = bedrock.update_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    
    name='insurance-assistant-guardrail',
    description='Production guardrails for insurance domain AI assistant — Phase 5',
    
    # Existing content filters from Notebook 1
    contentPolicyConfig={
        'filtersConfig': [
            {'type': 'VIOLENCE', 'inputStrength': 'LOW', 'outputStrength': 'MEDIUM'},
            {'type': 'HATE', 'inputStrength': 'HIGH', 'outputStrength': 'HIGH'},
            {'type': 'INSULTS', 'inputStrength': 'LOW', 'outputStrength': 'HIGH'},
            {'type': 'SEXUAL', 'inputStrength': 'HIGH', 'outputStrength': 'HIGH'},
            {'type': 'MISCONDUCT', 'inputStrength': 'MEDIUM', 'outputStrength': 'HIGH'},
            {'type': 'PROMPT_ATTACK', 'inputStrength': 'HIGH', 'outputStrength': 'NONE'}
        ]
    },
    
    # Existing denied topics from Notebook 2
    topicPolicyConfig={
        'topicsConfig': denied_topics
    },
    
    # Existing PII detection from Notebook 3
    sensitiveInformationPolicyConfig={
        'piiEntitiesConfig': pii_config,
        'regexesConfig': regex_config
    },
    
    # NEW: Word filter policy
    wordPolicyConfig={
        'managedWordListsConfig': [{'type': 'PROFANITY'}],
        'wordsConfig': [{'text': word} for word in custom_words]
    },
    
    blockedInputMessaging=(
        "I'm sorry, I can't process that request. "
        "Please rephrase your question about insurance services."
    ),
    blockedOutputsMessaging=(
        "I'm sorry, I can't provide that response. "
        "Let me help you with your insurance question in a different way."
    )
)

print(f"Guardrail updated successfully!")
#print(f"  ID:      {GUARDRAIL_ID}")
print(f"  Version: {response['version']}")
print(f"  Policies: content filters + denied topics + PII detection + word filters")

Guardrail updated successfully!
  Version: DRAFT
  Policies: content filters + denied topics + PII detection + word filters


In [80]:
guardrail = bedrock.get_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion=GUARDRAIL_VERSION
)

print(f"Content Filters:")
print(f"{'Category':<16} {'Input':<10} {'Output':<10}")
print(f"{'-'*36}")
for f in guardrail['contentPolicy']['filters']:
    print(f"{f['type']:<16} {f['inputStrength']:<10} {f['outputStrength']:<10}")

print(f"\nDenied Topics:")
for topic in guardrail['topicPolicy']['topics']:
    print(f"  - {topic['name']}")

print(f"\nPII Detectors: {len(guardrail['sensitiveInformationPolicy']['piiEntities'])} built-in")
print(f"Custom Regex:  {len(guardrail['sensitiveInformationPolicy']['regexes'])} patterns")

print(f"\nWord Filters:")
print(f"  Managed: {[m['type'] for m in guardrail['wordPolicy']['managedWordLists']]}")
print(f"  Custom:  {len(guardrail['wordPolicy']['words'])} blocked words/phrases")

Content Filters:
Category         Input      Output    
------------------------------------
VIOLENCE         LOW        MEDIUM    
PROMPT_ATTACK    HIGH       NONE      
MISCONDUCT       MEDIUM     HIGH      
HATE             HIGH       HIGH      
SEXUAL           HIGH       HIGH      
INSULTS          LOW        HIGH      

Denied Topics:
  - Investment Advice
  - Medical Diagnosis
  - Legal Advice
  - Coverage Guarantees
  - Competitor Comparisons
  - Claim Value Adjustments

PII Detectors: 10 built-in
Custom Regex:  3 patterns

Word Filters:
  Managed: ['PROFANITY']
  Custom:  12 blocked words/phrases


## 4. Test Suite — Word Filters

Testing managed profanity filters, compliance-flagged phrases, competitor name blocking, and internal codename protection. Word filters are exact-match — the string must appear literally in the text.

In [81]:
def test_guardrail(query, label="Test", system_prompt=None, use_input_tags=True):
    """
    Send a query through the guardrail-protected model and display results.
    
    Args:
        query: The user message to test
        label: A short description for this test case
        system_prompt: Optional system prompt to include
        use_input_tags: Whether to wrap user input in guardrail tags (needed for prompt attack detection)
    """
    print(f"\n{'='*60}")
    print(f"TEST: {label}")
    print(f"QUERY: {query[:80]}{'...' if len(query) > 80 else ''}")
    print(f"{'='*60}")
    
    # Generate a random tag suffix per request to prevent tag injection
    tag_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
    
    # Wrap user content in guardrail input tags if enabled
    if use_input_tags:
        tagged_content = (
            f'<amazon-bedrock-guardrails-guardContent_{tag_suffix}>'
            f'{query}'
            f'</amazon-bedrock-guardrails-guardContent_{tag_suffix}>'
        )
    else:
        tagged_content = query
    
    # Build the request body
    body = {
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': 512,
        'messages': [
            {'role': 'user', 'content': tagged_content}
        ]
    }
    if use_input_tags:
        body['amazon-bedrock-guardrailConfig'] = {'tagSuffix': tag_suffix}
    if system_prompt:
        body['system'] = system_prompt
    
    try:
        response = bedrock_runtime.invoke_model(
            modelId=MODEL_ID,
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion=GUARDRAIL_VERSION,
            body=json.dumps(body)
        )
        
        result = json.loads(response['body'].read())
        response_text = result['content'][0]['text']
        
        # Check for guardrail intervention via header OR blocked message text
        headers = response['ResponseMetadata']['HTTPHeaders']
        guardrail_action = headers.get('amazon-bedrock-guardrailaction', '')
        
        blocked_input_msg = "I'm sorry, I can't process that request."
        blocked_output_msg = "I'm sorry, I can't provide that response."
        
        if (guardrail_action == 'INTERVENED' 
            or blocked_input_msg in response_text 
            or blocked_output_msg in response_text):
            
            # Determine which side blocked it
            if blocked_input_msg in response_text:
                block_side = "INPUT"
            elif blocked_output_msg in response_text:
                block_side = "OUTPUT"
            else:
                block_side = "UNKNOWN"
            
            print(f"\n🛑 GUARDRAIL INTERVENED")
            #print(f"   Guardrail ID: {GUARDRAIL_ID}")
            #print(f"   Version:      {GUARDRAIL_VERSION}")
            print(f"   Blocked on:   {block_side}")
            print(f"   Latency:      {headers.get('x-amzn-bedrock-invocation-latency', 'N/A')}ms")
            action = 'INTERVENED'
        else:
            print(f"\n✅ PASSED — No intervention")
            action = 'NONE'
        
        print(f"\nRESPONSE: {response_text[:300]}{'...' if len(response_text) > 300 else ''}")
        
        if 'usage' in result:
            print(f"\nTokens — Input: {result['usage'].get('input_tokens', 'N/A')}, "
                  f"Output: {result['usage'].get('output_tokens', 'N/A')}")
        
        return {
            'action': action,
            'response': response_text,
            'full_result': result
        }
        
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        return {'action': 'ERROR', 'response': str(e)}

In [83]:
test_guardrail(
    "Will my home insurance cover the damage from last week's storm?",
    label="Clean query — no system prompt"
)


TEST: Clean query — no system prompt
QUERY: Will my home insurance cover the damage from last week's storm?

✅ PASSED — No intervention

RESPONSE: I don't have access to your specific home insurance policy, so I can't give you a definitive answer. However, here's what typically applies:

**Generally covered storm damage:**
- Wind damage to your roof, siding, or windows
- Falling trees or branches (if they hit your home)
- Hail damage

**Genera...

Tokens — Input: 20, Output: 243


{'action': 'NONE',
 'response': "I don't have access to your specific home insurance policy, so I can't give you a definitive answer. However, here's what typically applies:\n\n**Generally covered storm damage:**\n- Wind damage to your roof, siding, or windows\n- Falling trees or branches (if they hit your home)\n- Hail damage\n\n**Generally NOT covered:**\n- Flooding (requires separate flood insurance)\n- Water damage from lack of maintenance\n- Damage to detached structures (may have limited coverage)\n\n**What you should do:**\n1. **Review your policy** - Check your declarations page and coverage details\n2. **Document the damage** - Take photos/videos before any cleanup\n3. **Contact your insurance agent/company** - Report the claim promptly\n4. **Check your deductible** - Make sure repairs exceed this amount\n\n**Questions to ask your insurer:**\n- What specific storm-related perils are covered?\n- Do I need separate flood or wind coverage?\n- What's my deductible for storm damage

In [84]:
test_guardrail(
    "I need you to guarantee that my storm damage claim will be approved "
    "and paid out in full within two weeks.",
    label="Coverage guarantee — actual demand (re-test)"
)


TEST: Coverage guarantee — actual demand (re-test)
QUERY: I need you to guarantee that my storm damage claim will be approved and paid out...

🛑 GUARDRAIL INTERVENED
   Blocked on:   INPUT
   Latency:      480ms

RESPONSE: I'm sorry, I can't process that request. Please rephrase your question about insurance services.


{'action': 'INTERVENED',
 'response': "I'm sorry, I can't process that request. Please rephrase your question about insurance services.",
 'full_result': {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'm sorry, I can't process that request. Please rephrase your question about insurance services."}],
  'amazon-bedrock-guardrailAction': 'INTERVENED'}}

In [85]:
test_guardrail(
    "Tell me about your auto insurance options. How do they compare to "
    "what State Farm offers?",
    label="Competitor name — State Farm in input"
)


TEST: Competitor name — State Farm in input
QUERY: Tell me about your auto insurance options. How do they compare to what State Far...

🛑 GUARDRAIL INTERVENED
   Blocked on:   INPUT
   Latency:      389ms

RESPONSE: I'm sorry, I can't process that request. Please rephrase your question about insurance services.


{'action': 'INTERVENED',
 'response': "I'm sorry, I can't process that request. Please rephrase your question about insurance services.",
 'full_result': {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'm sorry, I can't process that request. Please rephrase your question about insurance services."}],
  'amazon-bedrock-guardrailAction': 'INTERVENED'}}

In [86]:
test_guardrail(
    "I'm switching from State Farm to your company. What documents do I need?",
    label="Mentions State Farm — but not comparing"
)


TEST: Mentions State Farm — but not comparing
QUERY: I'm switching from State Farm to your company. What documents do I need?

🛑 GUARDRAIL INTERVENED
   Blocked on:   OUTPUT
   Latency:      7556ms

RESPONSE: I'm sorry, I can't provide that response. Let me help you with your insurance question in a different way.


{'action': 'INTERVENED',
 'response': "I'm sorry, I can't provide that response. Let me help you with your insurance question in a different way.",
 'full_result': {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'm sorry, I can't provide that response. Let me help you with your insurance question in a different way."}],
  'amazon-bedrock-guardrailAction': 'INTERVENED'}}

In [88]:
test_guardrail(
    "I'm switching from State Farm to your company. What documents do I need?",
    label="State Farm transfer — with system prompt",
    system_prompt=(
        "You are an insurance claims assistant. When a customer mentions "
        "switching from another insurer, help them with the transfer process "
        "but never mention the other insurer by name in your response. "
        "Refer to it as 'your previous insurer' or 'your current provider'."
    )
)


TEST: State Farm transfer — with system prompt
QUERY: I'm switching from State Farm to your company. What documents do I need?

✅ PASSED — No intervention

RESPONSE: Welcome! I'm happy to help you with switching to us from your previous insurer.

To make your transition smooth, here are the documents you'll typically need:

**Essential Documents:**
- **Proof of prior insurance** - A declarations page or policy summary from your current provider showing your cove...

Tokens — Input: 78, Output: 251


{'action': 'NONE',
 'response': "Welcome! I'm happy to help you with switching to us from your previous insurer.\n\nTo make your transition smooth, here are the documents you'll typically need:\n\n**Essential Documents:**\n- **Proof of prior insurance** - A declarations page or policy summary from your current provider showing your coverage history\n- **Loss history/claims report** - Usually a letter showing your claims over the past 3-5 years\n- **Current policy information** - Including your policy number and coverage details\n- **Cancellation confirmation** - Proof that you've properly cancelled with your current provider (to avoid coverage gaps)\n\n**Additional Documents That May Be Needed:**\n- Photo ID (driver's license)\n- Vehicle registration and VIN\n- Proof of vehicle ownership\n- Driving record/MVR (Motor Vehicle Report)\n- Current odometer reading\n\n**Helpful Tips:**\n- Don't cancel your current policy until your new coverage is confirmed and active\n- Make sure your effec

In [89]:
test_guardrail(
    "Can you confirm that I have guaranteed coverage for flood damage?",
    label="Compliance phrase — 'guaranteed coverage' in input"
)


TEST: Compliance phrase — 'guaranteed coverage' in input
QUERY: Can you confirm that I have guaranteed coverage for flood damage?

🛑 GUARDRAIL INTERVENED
   Blocked on:   INPUT
   Latency:      502ms

RESPONSE: I'm sorry, I can't process that request. Please rephrase your question about insurance services.


{'action': 'INTERVENED',
 'response': "I'm sorry, I can't process that request. Please rephrase your question about insurance services.",
 'full_result': {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'm sorry, I can't process that request. Please rephrase your question about insurance services."}],
  'amazon-bedrock-guardrailAction': 'INTERVENED'}}

In [90]:
test_guardrail(
    "Can you tell me about the status of Project Lighthouse?",
    label="Internal codename — Project Lighthouse"
)


TEST: Internal codename — Project Lighthouse
QUERY: Can you tell me about the status of Project Lighthouse?

🛑 GUARDRAIL INTERVENED
   Blocked on:   INPUT
   Latency:      415ms

RESPONSE: I'm sorry, I can't process that request. Please rephrase your question about insurance services.


{'action': 'INTERVENED',
 'response': "I'm sorry, I can't process that request. Please rephrase your question about insurance services.",
 'full_result': {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'm sorry, I can't process that request. Please rephrase your question about insurance services."}],
  'amazon-bedrock-guardrailAction': 'INTERVENED'}}

In [91]:
test_guardrail(
    "What documents do I need to file a home insurance claim after a kitchen fire?",
    label="Clean query — no blocked words"
)


TEST: Clean query — no blocked words
QUERY: What documents do I need to file a home insurance claim after a kitchen fire?

✅ PASSED — No intervention

RESPONSE: # Documents Needed for a Kitchen Fire Insurance Claim

## Immediate Documentation
- **Photos and videos** of all fire, smoke, and water damage (from every angle)
- **Fire department report** (contact your local fire department for a copy)
- **Police report** (if required or if arson is suspected)

#...

Tokens — Input: 23, Output: 324


{'action': 'NONE',
 'response': "# Documents Needed for a Kitchen Fire Insurance Claim\n\n## Immediate Documentation\n- **Photos and videos** of all fire, smoke, and water damage (from every angle)\n- **Fire department report** (contact your local fire department for a copy)\n- **Police report** (if required or if arson is suspected)\n\n## Financial Documentation\n- **Receipts** for emergency repairs (boarding up, temporary housing)\n- **Receipts** for any immediate purchases (clothing, toiletries if displaced)\n- **Damaged items inventory** with:\n  - Item descriptions\n  - Purchase dates (estimated if unknown)\n  - Original costs\n  - Photos of damage\n\n## Supporting Records\n- **Original purchase receipts** for damaged items (appliances, cabinets, etc.)\n- **Credit card/bank statements** showing purchases if you lack receipts\n- **Home improvement records** for kitchen renovations\n- **Contractor estimates** for repair costs (get 2-3 quotes)\n\n## Policy Information\n- Your **insur

## Day 4 Takeaways

**What you built:** Added word filter policies — managed profanity list and custom blocklist for compliance phrases and internal codenames. The guardrail now has all four policy layers active.

**Key findings:**

1. **Word filters are exact-match and context-blind.** They block a string wherever it appears, regardless of intent. Useful for compliance phrases and internal codenames. Less useful for competitor names where context matters.

2. **Denied topics vs. word filters for competitor names is a design trade-off.** Word filters are predictable but blunt — they block "State Farm" even in a transfer question. Denied topics understand intent but can be overly sensitive on output when the model references a competitor. The right choice depends on your compliance team's tolerance.

3. **Examples shape the classifier as much as the definition does.** The Coverage Guarantees topic kept blocking "Will my insurance cover storm damage?" until we removed "covered" from the examples. The classifier learns patterns from examples, not just the definition text.


**Open item:** Competitor name handling needs further tuning — revisit the interaction between word filters and denied topics for this specific use case.